# Stage 8 — Verify (two-tier)

Rebuild the graph **from the edge list** (not from any saved graph object) and recompute every
headline number, comparing against `data/expected_values.json` written by Stage 6.

**Two tiers (D7):**
- **Exact tier** — deterministic results (persistent-predicate counts, first-after-recall, max gap,
  sensitivity subsets, DAG, depth). Must match to the integer.
- **Drift-tolerant tier** — quantities that move on a live re-fetch (corpus size, edge count,
  component sizes). Pass if within tolerance; both numbers printed so drift is visible.

Set `DATA_DIR` below. Network: none.

**Note (2026-09):** this notebook recomputes the headline numbers from the SAME run's
`data/*.csv` and compares them to the SAME run's `expected_values.json`. It is a self-consistency
check, not independent reproduction — the reproduction result is `python compare_to_reference.py`
run by a second analyst on the frozen inputs.

In [1]:
DATA_DIR = "./data"

In [2]:
import os, json
import pandas as pd
import networkx as nx

E = json.load(open(os.path.join(DATA_DIR, "expected_values.json")))
corpus = pd.read_csv(os.path.join(DATA_DIR, "corpus.csv"), dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
edges = pd.read_csv(os.path.join(DATA_DIR, "predicate_edges.csv"), dtype=str)
edges_rule1 = pd.read_csv(os.path.join(DATA_DIR, "predicate_edges_rule1.csv"), dtype=str)
recalled = pd.read_csv(os.path.join(DATA_DIR, "recalled_nodes.csv"), dtype=str)

year = dict(zip(corpus["k_number"], corpus["decision_year"]))
recall_date = dict(zip(recalled["k_number"], pd.to_datetime(recalled["event_date_initiated"], errors="coerce")))
recall_class = dict(zip(recalled["k_number"], recalled["worst_class"]))
recalled_set = set(recalled["k_number"]) & set(corpus["k_number"])

results = []
def check(name, expected, computed, tier="exact", tol=0.0):
    if tier == "exact":
        ok = (expected == computed)
    else:
        try:
            ok = abs(expected - computed) <= tol * max(1, abs(expected))
        except TypeError:
            ok = (expected == computed)
    tag = "PASS" if ok else ("PASS(drift)" if tier == "drift" and
           abs(expected-computed) <= 0.05*max(1,abs(expected)) else "FAIL")
    if tier == "drift" and not ok:
        tag = "FAIL"
    results.append(ok or (tier == "drift" and abs(expected-computed) <= 0.02*max(1,abs(expected))))
    flag = tag if (ok or tag.startswith("PASS")) else "FAIL"
    print(f"  [{flag:11}] {name:38} expected={expected!s:16} computed={computed}")

### Rebuild graph and recompute

In [3]:
G = nx.DiGraph(); G.add_nodes_from(corpus["k_number"])
for _, r in edges.iterrows():
    if r["predicate_knumber"] in G and r["device_knumber"] in G:
        G.add_edge(r["predicate_knumber"], r["device_knumber"], confidence=r["confidence"])

def analysis(edge_df, recalled_ok):
    H = nx.DiGraph(); H.add_nodes_from(corpus["k_number"])
    for _, r in edge_df.iterrows():
        if r["predicate_knumber"] in H and r["device_knumber"] in H:
            H.add_edge(r["predicate_knumber"], r["device_knumber"])
    preds, cits, downstream = set(), 0, set()
    for A in recalled_ok:
        if A not in H: continue
        rd = recall_date.get(A)
        if pd.isna(rd): continue
        for B in H.successors(A):
            if B in recalled_set: continue
            by = year.get(B)
            if pd.notna(by) and by > rd.year:
                preds.add(A); cits += 1; downstream.add(B)
    return [len(preds), cits, len(downstream)]

recalled_nodes = recalled_set & set(G.nodes())
connected = [n for n in G if G.degree(n) > 0]
largest_wcc = max((len(c) for c in nx.weakly_connected_components(G)), default=0)
depth = nx.dag_longest_path_length(nx.condensation(G))  # match Stage 6: depth on condensation (graph is a near-DAG)
base = analysis(edges, recalled_nodes)
cii = {A for A in recalled_nodes if recall_class.get(A) in ("Class I", "Class II")}
hc = edges[edges["confidence"] == "NEAR_CUE"]   # was SECTION_HEADED

first_after = 0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd): continue
    cy = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if cy and min(cy) > rd.year: first_after += 1
max_gap = 0.0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd): continue
    cy = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if cy: max_gap = max(max_gap, max(cy) - rd.year)

### Tier 1 — deterministic (exact)

In [4]:
print("=== deterministic tier (must match exactly) ===")
check("is a DAG", E["is_dag"], nx.is_directed_acyclic_graph(G))
check("max chain depth", E["max_chain_depth"], depth)
check("recalled nodes", E["recalled_nodes"], len(recalled_nodes))
check("persistent predicates", E["persistent_predicates"], base[0])
check("post-recall citations", E["post_recall_citations"], base[1])
check("downstream devices", E["downstream_devices"], base[2])
check("first cited after recall", E["first_after_recall"], first_after)
check("max recall-to-latest (yr)", E["max_gap_years"], round(max_gap, 1))
check("sensitivity class I/II", tuple(E["sens_class_I_II"]), tuple(analysis(edges, cii)))
check("sensitivity high-conf", tuple(E["sens_high_conf"]), tuple(analysis(hc, recalled_nodes)))
check("sensitivity combined", tuple(E["sens_combined"]), tuple(analysis(hc, cii)))
check("sensitivity maximal rule (R1)", tuple(E["sens_maximal_rule"]), tuple(analysis(edges_rule1, recalled_nodes)))
check("edges (Rule 1)", E["edges_rule1"], len(edges_rule1.drop_duplicates(subset=["predicate_knumber","device_knumber"])))

=== deterministic tier (must match exactly) ===
  [PASS       ] is a DAG                               expected=False            computed=False
  [PASS       ] max chain depth                        expected=30               computed=30
  [PASS       ] recalled nodes                         expected=1691             computed=1691
  [PASS       ] persistent predicates                  expected=921              computed=921
  [PASS       ] post-recall citations                  expected=2712             computed=2712
  [PASS       ] downstream devices                     expected=1798             computed=1798
  [PASS       ] first cited after recall               expected=382              computed=382
  [PASS       ] max recall-to-latest (yr)              expected=21               computed=21
  [PASS       ] sensitivity class I/II                 expected=(703, 1997, 1361) computed=(703, 1997, 1361)
  [PASS       ] sensitivity high-conf                  expected=(816, 2087, 1496) comput

### Tier 2 — drift-tolerant (live-refetch counts; ±2% ok)

In [5]:
print("\n=== formerly drift-tolerant; exact since the text corpus was frozen (D5) ===")
check("corpus devices", E["corpus_devices"], len(corpus))
check("edges", E["edges"], G.number_of_edges())
check("connected nodes", E["connected_nodes"], len(connected))
check("largest component", E["largest_component"], largest_wcc)


=== formerly drift-tolerant; exact since the text corpus was frozen (D5) ===
  [PASS       ] corpus devices                         expected=15581            computed=15581
  [PASS       ] edges                                  expected=29234            computed=29234
  [PASS       ] connected nodes                        expected=11960            computed=11960
  [PASS       ] largest component                      expected=11638            computed=11638


In [6]:
n_pass, n_tot = sum(results), len(results)
print("=" * 62)
print(f"RESULT: {n_pass}/{n_tot} checks passed")
assert n_pass == n_tot, f"{n_tot - n_pass} check(s) FAILED — see above"

RESULT: 17/17 checks passed
